# Difference in Differences

Difference in differences (DiD or DD) uses differences in over-time changes between outcomes in treated and control units to estimate causal effects of policy treatments. The differencing used in the design allows units to have different starting values (time invariant values). As long the units are moving in parallel (parallel trends assumption), then we can use the untreated units' differences as estimates of the counterfactual for the treated units' differences. The design thus provides an estimate of the ATT. 

In [ ]:
# install.packages('pacman') 

pacman::p_load(
   tidyverse,    # for data wrangling
   did,         # for conducting did anlaysis
   lfe,         # for fixed effects models
   modelsummary # for regression tables
)

DiD is probably the most popular impact evaluation design. For this example, we will use quarterly data on bankruptcies by counties in MI, OH, and PA. The data are from the [U.S. Bankruptcy Court filings](https://www.uscourts.gov/statistics-reports/caseload-statistics-data-tables). Bankruptcy filings are the outcome variable, measured as the number of non-business bankruptcy filings per 1000 residents in the county. 

We are interested in seeing if the adoption of online casinos and legalized online sports betting caused increased bankruptcy filings in Pennsylvania (which launched online gambling in 2019) and Michigan (which launched in 2021). The control group is Ohio, which did not have legalized online gambling until 2023. Our quarterly data runs from 2017 through 2022. We will also use population and unemployment data by county from the 2020 Census and the American Community Survey. 

In [ ]:
# read in the data from GitHub 
bankruptcy <- read_csv("https://raw.githubusercontent.com/bowendc/511_labs/refs/heads/main/bankruptcy.csv")

head(bankruptcy)

## $2 \times 2$ DiD

The classic $2 \times 2$ DiD design has two groups and two time periods. We can generate the DiD estimates using OLS in two different ways. First, we can use a treated unit dummy, a post-treatment dummy, and an interaction term to model. The interaction term represents the ATT. Here, let's focus on just the PA and OH comparisons (we'll deal with MI later).

In [ ]:
# estimate quick 2x2 dd model. Note that treat identifies all units who eventually
# adopt the election alignment with 1s, even prior to adoption. Post notes 
# all post-reform time periods.

# to create an interaction term, simply include the product of two variables: treat*post
# R will include both the constituent terms and the interaction term in the model.

# PA units get treated 
bankruptcy$treat <- if_else(bankruptcy$state_fips == "42", 1, 0)  

# all observations from quarter 11 on are post the treatment 
bankruptcy$post <- if_else(bankruptcy$time >= 11, 1, 0)

# felm is from the lfe package and let's us add fixed effects and/or clustered S.E. 
dd2x2 <- felm(bank.rate ~ treat*post
                | 0 | 0 | state_fips,     # value after first | shows var for FE (none here)
                                          # value after second | is for instrumental variables, leave as 0
                                          # value of third | is the variable on which you will calculate clustered S.E.s
                data = bankruptcy |> filter(state_fips!= "26" &
                                            time>=10 & time < 12) ) # drops all but two time periods and drops MI data

# view results. you could also use modelsummary
summary(dd)

The regression results show that PA has almost 1 filling per 1000 county residents lower, on average, than Ohio. Counties in OH and PA have just .0147 more filings per 1000 residents in quarter 11 than in quarter 10 (PA is treated in quarter 11). The DiD estimate is represented in `treat:post`: this is how the `post` estimate changes for PA compared to OH: -.0267. This is a very small estimate and in the wrong direction (online gambling adoption is associated with lower bankruptcy rates). 

But we have many more time periods than 2! We can incorporate them into our estimate using two-way fixed-effects (TWFE). 

To conduct a TWFE DiD model over multiple time periods, we need to recode the treatment variable. This time, we will code all PA counties at quarter 11 and later as being "treated". Then we will use fixed effects for time and state to capture time period and group differences, respectively. 

In [ ]:
# estimate fixed effects model (DD with multiple time periods)
# here, in the | time + state_fips | portion of the function,
# we specify the TWFE (time and state). The 0 part tells 
# R we are not using any instrumental variables. 
# the final | state_fips requests that we cluster standard 
# errors by state since we have repeated observations by 
# district over time. 

bankruptcy$treated = if_else(bankruptcy$state_fips == "42" & bankruptcy$time>=11, 1, 0)

ddTWFE.all <- felm(bank.rate ~ treated
                | time + state_fips | 0 | county_fips,
                data = bankruptcy |> filter(state_fips!= "26") )

summary(ddTWFE.all)

Ok, do we have evidence that Pennsylvania counties have higher bankruptcy filings after the adoption of legal online gambling? How do you know?

Let's just graph the relationship between bankruptcy and treatment status (just PA and OH), by quarter. The parallel trends assumption appears at least plausible at this point. 

In [ ]:
bankruptcy |> filter(state_fips != "26") |>
              group_by(treat, time) |> 
              summarize(mean = mean(bank.rate, na.rm = TRUE))  |>
  ggplot(aes(x = time, y = mean, color = factor(treat))) +
  geom_line()

We can use the `att_gt` function from the **{did}** package to improve our estimate. `att_gt` will calculate "doubly robust" DD estimates

In [ ]:
did <- att_gt(yname = "bank.rate",                    # outcome variable
              tname = "time",                         # time variable
              idname = "county_fips",                 # unit variable
              gname = "adopt",                        # variable that records first treated period by group
                                                      # OH counties coded 0 on this variable.
              allow_unbalanced_panel = TRUE,          # adjusts calculations for missing data in panel
              panel = TRUE,                           # the default is TRUE
              control_group = "nevertreated",         # alternative is 'notyettreated'
              data = bankruptcy |> filter(state_fips != "26") 
              )

summary(did)

The model output is best visualized using `ggdid` as an event study, which shows the difference between treatment and control units over time. 

In [ ]:
ggdid(did)

It takes a few quarters, but PA appears to see an increase in bankruptcy filings, compared to OH. 

We can average these estimates before and after treatment using `aggte`:

In [ ]:
aggte(did, type = "simple")

Here, we get a larger estimate of the effect of online gambling: .26 increase in filings per 1000 county residents. 

## DiD with differential timing (staggered timing)

The big benefit for using the **{did}** package is that we can handle staggered timing. Let's bring in our MI data.

In [ ]:
did.all <- att_gt(yname = "bank.rate",
                        tname = "time",
                        idname = "county_fips",
                        gname = "adopt",
                        allow_unbalanced_panel = TRUE,
                        panel = TRUE,
                        control_group = "nevertreated",
                        data = bankruptcy
                        )

summary(did.all)
ggdid(did.all)

Looks like our parallel trends assumption is less plausible for MI. Notice that we have two quarters with significantly different MI values from OH (Q7 and Q15). Not surprising, the p-value for pre-treatment parallel trends is significant, showing that we reject the null of parallel trends in the pre-treatment period. 

Let's work on improving our treatment and control county fit. First, we can focus just on the larger counties with more reliable data:

In [ ]:
did.big <- att_gt(yname = "bank.rate",
                        tname = "time",
                        idname = "county_fips",
                        gname = "adopt",
                        allow_unbalanced_panel = TRUE,
                        panel = TRUE,
                        control_group = "nevertreated",
                        data = bankruptcy |> filter(pop >= 100000)
                        )

summary(did.big)
ggdid(did.big)

and now let's include predictors on which we think the parallel trends assumption might depend. Here, we use the unemployment rate and the county population (logged) from 

In [ ]:
bankruptcy$lnpop <- log(bankruptcy$pop)
did.big2 <- att_gt(yname = "bank.rate",
                        tname = "time",
                        idname = "county_fips",
                        gname = "adopt",
                        allow_unbalanced_panel = TRUE,
                        panel = TRUE,
                        xformla = ~ unemployment + lnpop, 
                        control_group = "nevertreated",
                        data = bankruptcy |> filter(pop >= 100000)
                        )

summary(did.big2)
ggdid(did.big2)

We can store and than call up the ATT for the post-treatment periods, again using `aggte`.

In [ ]:
did.big2.simple <- aggte(did.big2, type = "simple")
did.big2.simple

In [ ]:
# the type = "dynamic" averages effects across groups by time since treated 
did.big2.dyn <- aggte(did.big2, type = "dynamic")
did.big2.dyn
ggdid(did.big2.dyn)



In [ ]:
# type = "group" generates ATT by group - in this case, the state
did.big2.gp <- aggte(did.big2, type = "group")
did.big2.gp
ggdid(did.big2.gp)

In [ ]:
# type = "calendar" generates ATT estimes by time period
did.big2.cal <- aggte(did.big2, type = "calendar")
did.big2.cal
ggdid(did.big2.cal)

## Placebo tests with DiD

Placebo tests are excellent ways to establish validity of your research design. The general idea is that you can utilize your same design, but in a situation where you would *not* expect to see an effect of the policy treatment. In this case, let's swap out our non-business bankruptcy filing rate for a *business* bankruptcy filing rate. We should not expect online gambling to increase business bankruptcies in MI and PA. 

In [ ]:
placebo <- att_gt(yname = "bus.bank.rate",
                        tname = "time",
                        idname = "county_fips",
                        gname = "adopt",
                        allow_unbalanced_panel = TRUE,
                        panel = TRUE,
                        xformla = ~ unemployment + pop,
                        control_group = "nevertreated",
                        data = bankruptcy |> filter(pop >= 100000)
                        )

placebo.agg <- aggte(placebo, type = "simple")
placebo.agg

In [ ]:
ggdid(placebo)

Sure enough, we find no evidence that online gambling increased business bankruptcy filing rates, and we do find evidence that it increased non-business bankruptcy filing rates per quarter. Let's present our ATTs in a quick table. 

In [ ]:
modelsummary(list(did.big2.agg, placebo.agg))